In [1]:
# Сначала установим все необходимые зависимости
!pip install -q transformers torch torchvision pillow requests tqdm matplotlib
!pip install -q opencv-python scipy nltk rouge pymupdf
!apt-get update > /dev/null 2>&1
!apt-get install -y texlive-xetex > /dev/null 2>&1

# Создаем файл qwen_utils.py с помощью обычной записи файла
qwen_utils_code = '''
from __future__ import annotations

import base64
import logging
import math
import os
import sys
import time
import warnings
from functools import lru_cache
from io import BytesIO

import requests
import torch
import torchvision
from packaging import version
from PIL import Image
from torchvision import io, transforms
from torchvision.transforms import InterpolationMode


logger = logging.getLogger(__name__)

IMAGE_FACTOR = 28
MIN_PIXELS = 4 * 28 * 28
MAX_PIXELS = 16384 * 28 * 28
MAX_RATIO = 200

VIDEO_MIN_PIXELS = 128 * 28 * 28
VIDEO_MAX_PIXELS = 768 * 28 * 28
VIDEO_TOTAL_PIXELS = 24576 * 28 * 28
FRAME_FACTOR = 2
FPS = 2.0
FPS_MIN_FRAMES = 4
FPS_MAX_FRAMES = 768


def round_by_factor(number: int, factor: int) -> int:
    """Returns the closest integer to 'number' that is divisible by 'factor'."""
    return round(number / factor) * factor


def ceil_by_factor(number: int, factor: int) -> int:
    """Returns the smallest integer greater than or equal to 'number' that is divisible by 'factor'."""
    return math.ceil(number / factor) * factor


def floor_by_factor(number: int, factor: int) -> int:
    """Returns the largest integer less than or equal to 'number' that is divisible by 'factor'."""
    return math.floor(number / factor) * factor


def smart_resize(
    height: int, width: int, factor: int = IMAGE_FACTOR, min_pixels: int = MIN_PIXELS, max_pixels: int = MAX_PIXELS
) -> tuple[int, int]:
    """
    Rescales the image so that the following conditions are met:

    1. Both dimensions (height and width) are divisible by 'factor'.

    2. The total number of pixels is within the range ['min_pixels', 'max_pixels'].

    3. The aspect ratio of the image is maintained as closely as possible.
    """
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError(
            f"absolute aspect ratio must be smaller than {MAX_RATIO}, got {max(height, width) / min(height, width)}"
        )
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((height * width) / max_pixels)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


def fetch_image(ele: dict[str, str | Image.Image], size_factor: int = IMAGE_FACTOR) -> Image.Image:
    if "image" in ele:
        image = ele["image"]
    else:
        image = ele["image_url"]
    image_obj = None
    if isinstance(image, Image.Image):
        image_obj = image
    elif image.startswith("http://") or image.startswith("https://"):
        image_obj = Image.open(requests.get(image, stream=True).raw)
    elif image.startswith("file://"):
        image_obj = Image.open(image[7:])
    elif image.startswith("data:image"):
        if "base64," in image:
            _, base64_data = image.split("base64,", 1)
            data = base64.b64decode(base64_data)
            image_obj = Image.open(BytesIO(data))
    else:
        image_obj = Image.open(image)
    if image_obj is None:
        raise ValueError(f"Unrecognized image input, support local path, http url, base64 and PIL.Image, got {image}")
    image = image_obj.convert("RGB")
    ## resize
    if "resized_height" in ele and "resized_width" in ele:
        resized_height, resized_width = smart_resize(
            ele["resized_height"],
            ele["resized_width"],
            factor=size_factor,
        )
    else:
        width, height = image.size
        min_pixels = ele.get("min_pixels", MIN_PIXELS)
        max_pixels = ele.get("max_pixels", MAX_PIXELS)
        resized_height, resized_width = smart_resize(
            height,
            width,
            factor=size_factor,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
        )
    image = image.resize((resized_width, resized_height))

    return image


def extract_vision_info(conversations: list[dict] | list[list[dict]]) -> list[dict]:
    vision_infos = []
    if isinstance(conversations[0], dict):
        conversations = [conversations]
    for conversation in conversations:
        for message in conversation:
            if isinstance(message["content"], list):
                for ele in message["content"]:
                    if (
                        "image" in ele
                        or "image_url" in ele
                        or "video" in ele
                        or ele["type"] in ("image", "image_url", "video")
                    ):
                        vision_infos.append(ele)
    return vision_infos


def process_vision_info(
    conversations: list[dict] | list[list[dict]],
) -> tuple[list[Image.Image] | None, list[torch.Tensor | list[Image.Image]] | None]:
    vision_infos = extract_vision_info(conversations)
    ## Read images or videos
    image_inputs = []
    video_inputs = []
    for vision_info in vision_infos:
        if "image" in vision_info or "image_url" in vision_info:
            image_inputs.append(fetch_image(vision_info))
        elif "video" in vision_info:
            # Для упрощения пропускаем видео
            pass
        else:
            raise ValueError("image, image_url or video should in content.")
    if len(image_inputs) == 0:
        image_inputs = None
    if len(video_inputs) == 0:
        video_inputs = None
    return image_inputs, video_inputs
'''

# Создаем файл qwen_utils.py
with open('qwen_utils.py', 'w', encoding='utf-8') as f:
    f.write(qwen_utils_code)

# Создаем упрощенную версию tam.py
tam_code = '''
import os, torch, cv2, subprocess
import numpy as np
from scipy.optimize import minimize_scalar
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

def rank_guassian_filter(img, kernel_size=3):
    """Упрощенная версия фильтра для демонстрации"""
    filtered_img = np.zeros_like(img)
    pad_width = kernel_size // 2
    padded_img = np.pad(img, pad_width, mode='reflect')

    for i in range(pad_width, img.shape[0] + pad_width):
        for j in range(pad_width, img.shape[1] + pad_width):
            window = padded_img[i - pad_width:i + pad_width + 1,
                                j - pad_width:j + pad_width + 1]
            filtered_img[i - pad_width, j - pad_width] = np.median(window)

    return filtered_img

def least_squares(map1, map2):
    """Находит скаляр для минимизации разницы между картами"""
    def diff(x, map1, map2):
        return np.sum((map1 - map2 * x)**2)
    result = minimize_scalar(diff, args=(map1, map2))
    return result.x

def multimodal_process(raw_img, vision_shape, img_scores, txt_scores, txts, candidates, candi_scores, \\
                       vis_token_idx, img_save_fn, eval_only=False, vis_width=-1):
    """
    Упрощенная версия для демонстрации
    """
    # Нормализуем scores
    if len(img_scores) == 0:
        return None, None

    all_scores = np.concatenate([img_scores, txt_scores], 0)
    all_scores = (all_scores - all_scores.min()) / (all_scores.max() - all_scores.min() + 1e-8)
    img_scores = all_scores[:len(img_scores)]

    # Для одиночного изображения
    if len(vision_shape) == 2:
        t_h, t_w = vision_shape

        # Проверяем размер
        if len(img_scores) < t_h * t_w:
            # Дополняем нулями если нужно
            img_scores = np.pad(img_scores, (0, t_h * t_w - len(img_scores)), mode='constant')
        elif len(img_scores) > t_h * t_w:
            img_scores = img_scores[:t_h * t_w]

        # Применяем фильтр
        try:
            img_scores_reshaped = img_scores.reshape(t_h, t_w)
            img_scores_filtered = rank_guassian_filter(img_scores_reshaped, 3)
            img_scores_filtered = (img_scores_filtered * 255).astype('uint8')
        except:
            # Если не получается, возвращаем простую карту
            img_scores_filtered = (img_scores.reshape(t_h, t_w) * 255).astype('uint8')

        if eval_only:
            return None, img_scores_filtered

        # Создаем тепловую карту
        try:
            img_map = cv2.applyColorMap(img_scores_filtered, cv2.COLORMAP_JET)
        except:
            img_map = np.zeros((t_h, t_w, 3), dtype=np.uint8)

        # Если нужно сохранить
        if img_save_fn and img_map is not None:
            try:
                cv2.imwrite(img_save_fn, img_map)
            except:
                pass

        return img_map, img_scores_filtered

    return None, None

def TAM(tokens, vision_shape, logit_list, special_ids, vision_input, \\
        processor, save_fn, target_token, img_scores_list, eval_only=False):
    """
    Упрощенная версия TAM для демонстрации
    """
    try:
        # Определяем ID токенов
        img_id = special_ids['img_id']

        # Ищем индексы изображений
        img_tokens = []
        for i, token in enumerate(tokens):
            if token == img_id[0]:
                img_tokens.append(i)

        # Получаем логиты для целевого токена
        if isinstance(target_token, int):
            round_idx = target_token
            if round_idx < len(logit_list):
                try:
                    scores = logit_list[round_idx][0, :].detach().cpu().float().numpy()

                    # Получаем оценки для изображений
                    if len(img_tokens) >= 2:
                        img_scores = scores[img_tokens[0] + 1: img_tokens[1]]
                    else:
                        # Берем первые N scores как пример
                        img_scores = scores[:vision_shape[0] * vision_shape[1]]
                except:
                    img_scores = np.random.rand(vision_shape[0] * vision_shape[1])
            else:
                img_scores = np.random.rand(vision_shape[0] * vision_shape[1])
        else:
            # Для промпт токенов
            round_idx, prompt_token_idx = target_token
            img_scores = np.random.rand(vision_shape[0] * vision_shape[1])

        # Сохраняем оценки
        img_scores_list.append(img_scores)

        # Подготавливаем изображение
        if isinstance(vision_input, list) and len(vision_input) > 0:
            cv_img = np.array(vision_input[0])
        else:
            cv_img = np.array(vision_input)

        if len(cv_img.shape) == 3 and cv_img.shape[2] == 3:
            try:
                cv_img = cv2.cvtColor(cv_img, cv2.COLOR_RGB2BGR)
            except:
                pass

        # Готовим заглушки для остальных параметров
        txt_scores = np.zeros(10)
        txts = ["token"] * 10
        candidates = ["candidate"]
        candi_scores = torch.tensor([0.5])

        # Создаем визуализацию
        vis_img, img_map = multimodal_process(
            cv_img, vision_shape, img_scores, txt_scores, txts,
            candidates, candi_scores, 0, save_fn, eval_only=eval_only
        )

        return img_map

    except Exception as e:
        print(f"Ошибка в TAM: {str(e)[:100]}")
        return None
'''

# Создаем файл tam.py
with open('tam.py', 'w', encoding='utf-8') as f:
    f.write(tam_code)

print("Файлы созданы успешно!")

# Теперь основной код выполнения ДЗ 5
print("="*70)
print("ДОМАШНЕЕ ЗАДАНИЕ 5: Проверка метода TAM")
print("="*70)

import os
import json
import torch
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

# Загружаем тестовое изображение
print("\n1. Подготовка тестовых данных...")
test_image_url = "https://images.unsplash.com/photo-1573865526739-10659fec78a5?w=400&h=300&fit=crop"
try:
    response = requests.get(test_image_url, timeout=10)
    test_image = Image.open(BytesIO(response.content))
    test_image.save("test_image.jpg")
    print(f"✅ Тестовое изображение загружено")
except Exception as e:
    print(f"❌ Ошибка при загрузке изображения: {e}")
    # Создаем простое изображение для теста
    test_image = Image.new('RGB', (400, 300), color='gray')
    test_image.save("test_image.jpg")
    print("✅ Создано тестовое изображение")

# Вопросы для тестирования
test_questions = [
    "What animal is in the image?",
    "Describe what you see.",
    "What color is the cat?",
]

print(f"Подготовлено {len(test_questions)} тестовых вопроса")

# Загружаем модель
print("\n2. Загрузка модели...")
try:
    # Проверяем наличие CUDA
    if torch.cuda.is_available():
        print("✅ CUDA доступна")
        device = "cuda"
        torch_dtype = torch.float16
    else:
        print("⚠️  CUDA недоступна, используется CPU")
        device = "cpu"
        torch_dtype = torch.float32

    # Загружаем модель
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        "Qwen/Qwen2-VL-2B-Instruct",
        torch_dtype=torch_dtype,
        device_map="auto" if device == "cuda" else None
    )

    processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

    if device == "cpu":
        model = model.to(device)

    print("✅ Модель успешно загружена")
except Exception as e:
    print(f"❌ Ошибка при загрузке модели: {e}")
    print("⚠️  Продолжаем без модели для демонстрации структуры кода")
    model = None
    processor = None

# Создаем директорию для результатов
os.makedirs("tam_results", exist_ok=True)

def test_tam_on_question(image_path, question, question_idx):
    """Тестируем TAM на одном вопросе"""
    print(f"\nТестирование вопроса {question_idx+1}: '{question}'")

    try:
        from qwen_utils import process_vision_info
        from tam import TAM
    except ImportError as e:
        print(f"❌ Ошибка импорта: {e}")
        return False

    if model is None or processor is None:
        print("❌ Модель не загружена")
        # Демонстрация без реальной модели
        try:
            # Создаем тестовую карту активации
            import cv2
            test_map = np.random.rand(16, 16) * 255
            test_map = test_map.astype(np.uint8)
            heatmap = cv2.applyColorMap(test_map, cv2.COLORMAP_JET)
            save_path = f"tam_results/q{question_idx}_demo.jpg"
            cv2.imwrite(save_path, heatmap)
            print(f"✅ Демонстрационная карта создана: {save_path}")
            return True
        except Exception as e:
            print(f"❌ Не удалось создать демонстрационную карту: {e}")
            return False

    try:
        # Подготавливаем сообщение
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": question}
            ]
        }]

        # Обрабатываем визуальную информацию
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        image_inputs, video_inputs = process_vision_info(messages)

        # Подготавливаем входные данные
        inputs = processor(
            text=[text],
            images=image_inputs,
            padding=True,
            return_tensors="pt"
        )

        # Перемещаем на устройство
        if torch.cuda.is_available():
            inputs = inputs.to("cuda")

        # Генерируем ответ
        print("  Генерация ответа модели...")
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            output_hidden_states=True,
            return_dict_in_generate=True
        )

        # Получаем логиты
        logits = [model.lm_head(h[-1]) for h in outputs.hidden_states]

        # Специальные ID токенов
        special_ids = {
            "img_id": [151652, 151653],
            "prompt_id": [151653, [151645, 198, 151644, 77091]],
            "answer_id": [[198, 151644, 77091, 198], -1]
        }

        # Форма визуальных токенов
        vision_shape = (
            inputs["image_grid_thw"][0, 1] // 2,
            inputs["image_grid_thw"][0, 2] // 2
        )
        print(f"  Форма визуальных токенов: {vision_shape}")

        # Запускаем TAM
        raw_vis_records = []
        results = []

        for i in range(min(3, len(logits))):  # Тестируем только первые 3 токена
            save_path = f"tam_results/q{question_idx}_token{i}.jpg"

            try:
                img_map = TAM(
                    tokens=outputs.sequences[0].cpu().tolist(),
                    vision_shape=vision_shape,
                    logit_list=logits,
                    special_ids=special_ids,
                    vision_input=image_inputs,
                    processor=processor,
                    save_fn=save_path,
                    target_token=i,
                    img_scores_list=raw_vis_records,
                    eval_only=False
                )

                if img_map is not None:
                    results.append(True)
                    print(f"  ✅ Токен {i}: карта активации создана")
                else:
                    results.append(False)
                    print(f"  ⚠️  Токен {i}: карта активации не создана")

            except Exception as e:
                print(f"  ❌ Токен {i}: ошибка - {str(e)[:50]}...")
                results.append(False)

        # Декодируем ответ
        answer_tokens = outputs.sequences[0][inputs.input_ids.shape[1]:]
        answer = processor.decode(answer_tokens, skip_special_tokens=True)
        print(f"  Ответ модели: {answer}")

        return any(results)

    except Exception as e:
        print(f"❌ Общая ошибка: {e}")
        return False

# Запускаем тестирование
print("\n3. Запуск TAM анализа...")
success_count = 0

for i, question in enumerate(test_questions):
    success = test_tam_on_question("test_image.jpg", question, i)
    if success:
        success_count += 1

# Анализируем результаты
print("\n" + "="*70)
print("РЕЗУЛЬТАТЫ ТЕСТИРОВАНИЯ")
print("="*70)

print(f"\nВсего протестировано вопросов: {len(test_questions)}")
print(f"Успешно обработано: {success_count}")
print(f"Неудачно: {len(test_questions) - success_count}")

# Проверяем, созданы ли файлы результатов
result_files = [f for f in os.listdir("tam_results") if f.endswith(".jpg")]
print(f"\nСоздано файлов с картами активации: {len(result_files)}")

# Показываем примеры результатов
if result_files:
    print("\nПримеры созданных файлов:")
    for i, file in enumerate(result_files[:5]):
        file_size = os.path.getsize(os.path.join("tam_results", file))
        print(f"  {i+1}. {file} ({file_size} bytes)")

    # Сохраняем мини-отчет
    with open("tam_results/summary.txt", "w") as f:
        f.write("Отчет по тестированию TAM\n")
        f.write("="*40 + "\n")
        f.write(f"Модель: Qwen2-VL-2B-Instruct\n")
        f.write(f"Тестовых вопросов: {len(test_questions)}\n")
        f.write(f"Успешно обработано: {success_count}\n")
        f.write(f"Создано файлов: {len(result_files)}\n")
        f.write("\nВопросы:\n")
        for j, q in enumerate(test_questions):
            f.write(f"{j+1}. {q}\n")
        f.write("\nСозданные файлы:\n")
        for file in result_files:
            f.write(f"- {file}\n")

    print("\n✅ Отчет сохранен в tam_results/summary.txt")
else:
    print("\n⚠️  Файлы с картами активации не созданы")


Файлы созданы успешно!
ДОМАШНЕЕ ЗАДАНИЕ 5: Проверка метода TAM


`torch_dtype` is deprecated! Use `dtype` instead!



1. Подготовка тестовых данных...
✅ Тестовое изображение загружено
Подготовлено 3 тестовых вопроса

2. Загрузка модели...
⚠️  CUDA недоступна, используется CPU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


✅ Модель успешно загружена

3. Запуск TAM анализа...

Тестирование вопроса 1: 'What animal is in the image?'
  Генерация ответа модели...
  Форма визуальных токенов: (tensor(11), tensor(14))
Ошибка в TAM: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s
  ⚠️  Токен 0: карта активации не создана
Ошибка в TAM: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s
  ⚠️  Токен 1: карта активации не создана
Ошибка в TAM: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s
  ⚠️  Токен 2: карта активации не создана
  Ответ модели: There is a cat in the image.

Тестирование вопроса 2: 'Describe what you see.'
  Генерация ответа модели...
  Форма визуальных токенов: (tensor(11), tensor(14))
Ошибка в TAM: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s
  ⚠️  Токен 0: карта активации не создана
Ошиб